In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

digit_recognizer_path = kagglehub.competition_download('digit-recognizer')

print('Data source import complete.')


100%|██████████| 15.3M/15.3M [00:00<00:00, 63.1MB/s]

Extracting files...


Data source import complete.


In [ ]:
import pandas as pd
import numpy as np

np.random.seed(1212)

import keras
from keras.models import Model
from keras.layers import *
from keras import optimizers

In [ ]:
df_train = pd.read_csv(f'{digit_recognizer_path}/train.csv')
df_test = pd.read_csv(f'{digit_recognizer_path}/test.csv')

In [ ]:
df_train.head() # 784 features, 1 label

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_features = df_train.iloc[:, 1:785]
df_label = df_train.iloc[:, 0]

X_test = df_test.iloc[:, 0:784]

print(X_test.shape)

(28000, 784)


In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_cv, y_train, y_cv = train_test_split(df_features, df_label,
                                                test_size = 0.2,
                                                random_state = 1212)

X_train = X_train.to_numpy().reshape(33600, 784) #(33600, 784)
X_cv = X_cv.to_numpy().reshape(8400, 784) #(8400, 784)

# Re-initialize X_test from df_test to ensure it's a DataFrame before conversion
X_test = df_test.to_numpy().reshape(28000, 784)

In [ ]:
print((min(X_train[1]), max(X_train[1])))

(np.int64(0), np.int64(255))


In [ ]:
# Feature Normalization
X_train = X_train.astype('float32'); X_cv= X_cv.astype('float32'); X_test = X_test.astype('float32')
X_train /= 255; X_cv /= 255; X_test /= 255

# Convert labels to One Hot Encoded
num_digits = 10
y_train = keras.utils.to_categorical(y_train, num_digits)
y_cv = keras.utils.to_categorical(y_cv, num_digits)

In [ ]:
# Printing 2 examples of labels after conversion
print(y_train[0]) # 2
print(y_train[3]) # 7

[0. 0. 1. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 1. 0. 0.]


In [ ]:
# Input Parameters
n_input = 784 # number of features
n_hidden_1 = 300
n_hidden_2 = 100
n_hidden_3 = 100
n_hidden_4 = 200
num_digits = 10

In [ ]:
Inp = Input(shape=(784,))
x = Dense(n_hidden_1, activation='relu', name = "Hidden_Layer_1")(Inp)
x = Dense(n_hidden_2, activation='relu', name = "Hidden_Layer_2")(x)
x = Dense(n_hidden_3, activation='relu', name = "Hidden_Layer_3")(x)
x = Dense(n_hidden_4, activation='relu', name = "Hidden_Layer_4")(x)
output = Dense(num_digits, activation='softmax', name = "Output_Layer")(x)

In [ ]:
# Our model would have '6' layers - input layer, 4 hidden layer and 1 output layer
model = Model(Inp, output)
model.summary() # We have 297,910 parameters to estimate

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_1 (Dense)          │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_2 (Dense)          │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_3 (Dense)          │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_4 (Dense)          │ (None, 200)            │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 297,910 (1.14 MB)

 Trainable params: 297,910 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Insert Hyperparameters
learning_rate = 0.1
training_epochs = 20
batch_size = 100
sgd = optimizers.SGD(learning_rate=learning_rate)

In [ ]:
# We rely on the plain vanilla Stochastic Gradient Descent as our optimizing methodology
model.compile(loss='categorical_crossentropy',
              optimizer='sgd',
              metrics=['accuracy'])

In [ ]:
history1 = model.fit(X_train, y_train,
                     batch_size = batch_size,
                     epochs = training_epochs,
                     verbose = 2,
                     validation_data=(X_cv, y_cv))

Epoch 1/20
336/336 - 3s - 8ms/step - accuracy: 0.9683 - loss: 0.1074 - val_accuracy: 0.9581 - val_loss: 0.1375
Epoch 2/20
336/336 - 3s - 9ms/step - accuracy: 0.9704 - loss: 0.1030 - val_accuracy: 0.9602 - val_loss: 0.1386
Epoch 3/20
336/336 - 2s - 6ms/step - accuracy: 0.9719 - loss: 0.0985 - val_accuracy: 0.9614 - val_loss: 0.1317
Epoch 4/20
336/336 - 2s - 6ms/step - accuracy: 0.9730 - loss: 0.0936 - val_accuracy: 0.9620 - val_loss: 0.1282
Epoch 5/20
336/336 - 2s - 6ms/step - accuracy: 0.9743 - loss: 0.0895 - val_accuracy: 0.9617 - val_loss: 0.1267
Epoch 6/20
336/336 - 2s - 6ms/step - accuracy: 0.9757 - loss: 0.0860 - val_accuracy: 0.9625 - val_loss: 0.1301
Epoch 7/20
336/336 - 3s - 10ms/step - accuracy: 0.9769 - loss: 0.0819 - val_accuracy: 0.9643 - val_loss: 0.1232
Epoch 8/20
336/336 - 2s - 7ms/step - accuracy: 0.9779 - loss: 0.0782 - val_accuracy: 0.9649 - val_loss: 0.1198
Epoch 9/20
336/336 - 2s - 6ms/step - accuracy: 0.9793 - loss: 0.0746 - val_accuracy: 0.9662 - val_loss: 0.1165


In [ ]:
Inp = Input(shape=(784,))
x = Dense(n_hidden_1, activation='relu', name = "Hidden_Layer_1")(Inp)
x = Dense(n_hidden_2, activation='relu', name = "Hidden_Layer_2")(x)
x = Dense(n_hidden_3, activation='relu', name = "Hidden_Layer_3")(x)
x = Dense(n_hidden_4, activation='relu', name = "Hidden_Layer_4")(x)
output = Dense(num_digits, activation='softmax', name = "Output_Layer")(x)

# We rely on ADAM as our optimizing methodology
adam = keras.optimizers.Adam(learning_rate=learning_rate)
model2 = Model(Inp, output)

model2.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
history2 = model2.fit(X_train, y_train,
                      batch_size = batch_size,
                      epochs = training_epochs,
                      verbose = 2,
                      validation_data=(X_cv, y_cv))

Epoch 1/20
336/336 - 6s - 17ms/step - accuracy: 0.8983 - loss: 0.3353 - val_accuracy: 0.9552 - val_loss: 0.1514
Epoch 2/20
336/336 - 4s - 11ms/step - accuracy: 0.9629 - loss: 0.1187 - val_accuracy: 0.9636 - val_loss: 0.1200
Epoch 3/20
336/336 - 3s - 8ms/step - accuracy: 0.9750 - loss: 0.0784 - val_accuracy: 0.9601 - val_loss: 0.1275
Epoch 4/20
336/336 - 3s - 9ms/step - accuracy: 0.9835 - loss: 0.0545 - val_accuracy: 0.9700 - val_loss: 0.1077
Epoch 5/20
336/336 - 4s - 11ms/step - accuracy: 0.9849 - loss: 0.0455 - val_accuracy: 0.9655 - val_loss: 0.1283
Epoch 6/20
336/336 - 4s - 12ms/step - accuracy: 0.9872 - loss: 0.0392 - val_accuracy: 0.9751 - val_loss: 0.0964
Epoch 7/20
336/336 - 3s - 8ms/step - accuracy: 0.9907 - loss: 0.0275 - val_accuracy: 0.9740 - val_loss: 0.0997
Epoch 8/20
336/336 - 3s - 10ms/step - accuracy: 0.9902 - loss: 0.0293 - val_accuracy: 0.9689 - val_loss: 0.1176
Epoch 9/20
336/336 - 3s - 10ms/step - accuracy: 0.9928 - loss: 0.0204 - val_accuracy: 0.9713 - val_loss: 0.

In [ ]:
Inp = Input(shape=(784,))
x = Dense(n_hidden_1, activation='relu', name = "Hidden_Layer_1")(Inp)
x = Dense(n_hidden_2, activation='relu', name = "Hidden_Layer_2")(x)
x = Dense(n_hidden_3, activation='relu', name = "Hidden_Layer_3")(x)
x = Dense(n_hidden_4, activation='relu', name = "Hidden_Layer_4")(x)
output = Dense(num_digits, activation='softmax', name = "Output_Layer")(x)

learning_rate = 0.01
adam = keras.optimizers.Adam(learning_rate=learning_rate)
model2a = Model(Inp, output)

model2a.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
history2a = model2a.fit(X_train, y_train,
                        batch_size = batch_size,
                        epochs = training_epochs,
                        verbose = 2,
                        validation_data=(X_cv, y_cv))

Epoch 1/20
336/336 - 5s - 16ms/step - accuracy: 0.9012 - loss: 0.3384 - val_accuracy: 0.9557 - val_loss: 0.1478
Epoch 2/20
336/336 - 5s - 14ms/step - accuracy: 0.9626 - loss: 0.1200 - val_accuracy: 0.9633 - val_loss: 0.1218
Epoch 3/20
336/336 - 3s - 9ms/step - accuracy: 0.9757 - loss: 0.0801 - val_accuracy: 0.9679 - val_loss: 0.1085
Epoch 4/20
336/336 - 4s - 12ms/step - accuracy: 0.9810 - loss: 0.0596 - val_accuracy: 0.9696 - val_loss: 0.1099
Epoch 5/20
336/336 - 3s - 8ms/step - accuracy: 0.9860 - loss: 0.0442 - val_accuracy: 0.9711 - val_loss: 0.1038
Epoch 6/20
336/336 - 3s - 8ms/step - accuracy: 0.9885 - loss: 0.0366 - val_accuracy: 0.9750 - val_loss: 0.0883
Epoch 7/20
336/336 - 6s - 18ms/step - accuracy: 0.9910 - loss: 0.0274 - val_accuracy: 0.9719 - val_loss: 0.1084
Epoch 8/20
336/336 - 3s - 9ms/step - accuracy: 0.9907 - loss: 0.0290 - val_accuracy: 0.9744 - val_loss: 0.1018
Epoch 9/20
336/336 - 3s - 8ms/step - accuracy: 0.9939 - loss: 0.0209 - val_accuracy: 0.9731 - val_loss: 0.11

Model 2B

In [ ]:
Inp = Input(shape=(784,))
x = Dense(n_hidden_1, activation='relu', name = "Hidden_Layer_1")(Inp)
x = Dense(n_hidden_2, activation='relu', name = "Hidden_Layer_2")(x)
x = Dense(n_hidden_3, activation='relu', name = "Hidden_Layer_3")(x)
x = Dense(n_hidden_4, activation='relu', name = "Hidden_Layer_4")(x)
output = Dense(num_digits, activation='softmax', name = "Output_Layer")(x)

learning_rate = 0.5
adam = keras.optimizers.Adam(learning_rate=learning_rate)
model2b = Model(Inp, output)

model2b.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
history2b = model2b.fit(X_train, y_train,
                        batch_size = batch_size,
                        epochs = training_epochs,
                            validation_data=(X_cv, y_cv))

Epoch 1/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.8990 - loss: 0.3349 - val_accuracy: 0.9543 - val_loss: 0.1556
Epoch 2/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9607 - loss: 0.1225 - val_accuracy: 0.9601 - val_loss: 0.1309
Epoch 3/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9735 - loss: 0.0825 - val_accuracy: 0.9699 - val_loss: 0.1021
Epoch 4/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9813 - loss: 0.0588 - val_accuracy: 0.9706 - val_loss: 0.0976
Epoch 5/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9869 - loss: 0.0418 - val_accuracy: 0.9758 - val_loss: 0.0866
Epoch 6/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9899 - loss: 0.0331 - val_accuracy: 0.9756 - val_loss: 0.0970
Epoch 7/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - accuracy: 0.9897 - loss: 0.0304 - val_accuracy: 0.9761 - val_loss: 0.0873
Epoch 8/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9905 - loss: 0.0286 - val_accuracy: 

In [ ]:
# Input Parameters
n_input = 784 # number of features
n_hidden_1 = 300
n_hidden_2 = 100
n_hidden_3 = 100
n_hidden_4 = 100
n_hidden_5 = 200
num_digits = 10

In [ ]:
Inp = Input(shape=(784,))
x = Dense(n_hidden_1, activation='relu', name = "Hidden_Layer_1")(Inp)
x = Dense(n_hidden_2, activation='relu', name = "Hidden_Layer_2")(x)
x = Dense(n_hidden_3, activation='relu', name = "Hidden_Layer_3")(x)
x = Dense(n_hidden_4, activation='relu', name = "Hidden_Layer_4")(x)
x = Dense(n_hidden_5, activation='relu', name = "Hidden_Layer_5")(x)
output = Dense(num_digits, activation='softmax', name = "Output_Layer")(x)

In [ ]:
# Our model would have '7' layers - input layer, 5 hidden layer and 1 output layer
model3 = Model(Inp, output)
model3.summary() # We have 308,010 parameters to estimate

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_8 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_1 (Dense)          │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_2 (Dense)          │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_3 (Dense)          │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_4 (Dense)          │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_5 (Dense)          │ (None, 200)            │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 308,010 (1.17 MB)

 Trainable params: 308,010 (1.17 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# We rely on 'Adam' as our optimizing methodology
adam = keras.optimizers.Adam(learning_rate=0.01)

model3.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
history3 = model3.fit(X_train, y_train,
                      batch_size = batch_size,
                      epochs = training_epochs,
                      validation_data=(X_cv, y_cv))

Epoch 1/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8940 - loss: 0.3471 - val_accuracy: 0.9475 - val_loss: 0.1766
Epoch 2/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9594 - loss: 0.1264 - val_accuracy: 0.9512 - val_loss: 0.1569
Epoch 3/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9718 - loss: 0.0887 - val_accuracy: 0.9664 - val_loss: 0.1134
Epoch 4/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9793 - loss: 0.0645 - val_accuracy: 0.9711 - val_loss: 0.1010
Epoch 5/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9841 - loss: 0.0500 - val_accuracy: 0.9700 - val_loss: 0.1037
Epoch 6/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9873 - loss: 0.0399 - val_accuracy: 0.9720 - val_loss: 0.0985
Epoch 7/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 12ms/step - accuracy: 0.9890 - loss: 0.0338 - val_accuracy: 0.9718 - val_loss: 0.1177
Epoch 8/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 10ms/step - accuracy: 0.9890 - loss: 0.0374 - val_accuracy

In [ ]:
# Input Parameters
n_input = 784 # number of features
n_hidden_1 = 300
n_hidden_2 = 100
n_hidden_3 = 100
n_hidden_4 = 200
num_digits = 10

In [ ]:
Inp = Input(shape=(784,))
x = Dense(n_hidden_1, activation='relu', name = "Hidden_Layer_1")(Inp)
x = Dropout(0.3)(x)
x = Dense(n_hidden_2, activation='relu', name = "Hidden_Layer_2")(x)
x = Dropout(0.3)(x)
x = Dense(n_hidden_3, activation='relu', name = "Hidden_Layer_3")(x)
x = Dropout(0.3)(x)
x = Dense(n_hidden_4, activation='relu', name = "Hidden_Layer_4")(x)
output = Dense(num_digits, activation='softmax', name = "Output_Layer")(x)

In [ ]:
# Our model would have '6' layers - input layer, 4 hidden layer and 1 output layer
model4 = Model(Inp, output)
model4.summary() # We have 297,910 parameters to estimate

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_9 (InputLayer)      │ (None, 784)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_1 (Dense)          │ (None, 300)            │       235,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_2 (Dense)          │ (None, 100)            │        30,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_3 (Dense)          │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Hidden_Layer_4 (Dense)          │ (None, 200)            │        20,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ Output_Layer (Dense)            │ (None, 10)             │         2,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 297,910 (1.14 MB)

 Trainable params: 297,910 (1.14 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
model4.compile(loss='categorical_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
history = model4.fit(X_train, y_train,
                    batch_size = batch_size,
                    epochs = training_epochs,
                    validation_data=(X_cv, y_cv))

Epoch 1/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - accuracy: 0.8113 - loss: 0.5892 - val_accuracy: 0.9412 - val_loss: 0.1916
Epoch 2/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9335 - loss: 0.2293 - val_accuracy: 0.9552 - val_loss: 0.1517
Epoch 3/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9483 - loss: 0.1755 - val_accuracy: 0.9635 - val_loss: 0.1201
Epoch 4/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9582 - loss: 0.1440 - val_accuracy: 0.9699 - val_loss: 0.1076
Epoch 5/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - accuracy: 0.9632 - loss: 0.1246 - val_accuracy: 0.9705 - val_loss: 0.1119
Epoch 6/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - accuracy: 0.9676 - loss: 0.1092 - val_accuracy: 0.9721 - val_loss: 0.1050
Epoch 7/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.9715 - loss: 0.0968 - val_accuracy: 0.9736 - val_loss: 0.0976
Epoch 8/20
336/336 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - accuracy: 0.9722 - loss: 0.0921 - val_accuracy:

In [ ]:
test_pred = pd.DataFrame(model4.predict(X_test, batch_size=200))
test_pred = pd.DataFrame(test_pred.idxmax(axis = 1))
test_pred.index.name = 'ImageId'
test_pred = test_pred.rename(columns = {0: 'Label'}).reset_index()
test_pred['ImageId'] = test_pred['ImageId'] + 1

test_pred.head()

140/140 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


,ImageId,Label
0,1,2
1,2,0
2,3,9
3,4,0
4,5,3


In [ ]:
test_pred.to_csv('mnist_submission.csv', index = False)

Using this model, we are able to achieve a score of 0.976, which places us at the top 55th percentile!